# InfluxDB Data Query Example

This notebook demonstrates how to connect to the local InfluxDB instance, query data using Flux, and display it using Pandas.

## Prerequisites
- Ensure Docker containers are running (`docker-compose up -d`)
- Install requirements: `pip install influxdb-client pandas matplotlib`

In [ ]:
import pandas as pd
from influxdb_client import InfluxDBClient
import warnings
from influxdb_client.client.warnings import MissingPivotFunction

# Suppress pivot warnings for clean output
warnings.simplefilter("ignore", MissingPivotFunction)

## 1. Connect to InfluxDB
We use the same credentials defined in `docker-compose.yml`.

In [ ]:
url = "http://localhost:8086"
token = "my-super-secret-auth-token"
org = "grafana_org"

client = InfluxDBClient(url=url, token=token, org=org)
query_api = client.query_api()

## 2. Query Global Recap Data
Fetch a summary table of all jobs, similar to the Global Dashboard.

In [ ]:
recap_query = '''
from(bucket: "recap_bucket")
  |> range(start: -365d)
  |> filter(fn: (r) => r["_measurement"] == "recap_data")
  |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> drop(columns: ["_start", "_stop", "_measurement"])
'''

recap_df = query_api.query_data_frame(recap_query)
# Clean up index if needed
if '_time' in recap_df.columns:
    recap_df['_time'] = pd.to_datetime(recap_df['_time'])

recap_df.head()

## 3. Query Detailed Job Data
Select a specific Job ID from the recap data and fetch its detailed timeseries.

In [ ]:
# Pick the first job_id from the recap data
if not recap_df.empty:
    job_id = recap_df['job_id'].iloc[0]
    print(f"Querying details for Job ID: {job_id}")

    # Note: Use 'JobId' (PascalCase) for run_data filtering as per schema
    job_query = f'''
    from(bucket: "grafana_bucket")
      |> range(start: -365d)
      |> filter(fn: (r) => r["_measurement"] == "run_data")
      |> filter(fn: (r) => r["JobId"] == "{job_id}")
      |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
      |> drop(columns: ["_start", "_stop", "_measurement"])
    '''

    job_df = query_api.query_data_frame(job_query)
    display(job_df.head())
else:
    print("No recap data found to traverse.")

## 4. Simple Visualization
Plotting Main Drive Power for the selected job.

In [ ]:
import matplotlib.pyplot as plt

if 'job_df' in locals() and not job_df.empty and 'Active_ActProcVal_All_MainDrive_ActPower' in job_df.columns:
    plt.figure(figsize=(10, 6))
    plt.plot(job_df['_time'], job_df['Active_ActProcVal_All_MainDrive_ActPower'], label='Main Drive Power')
    plt.title(f'Power Consumption for Job {job_id}')
    plt.xlabel('Time')
    plt.ylabel('Power')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Data for plotting unavailable.")